# 🛡️ SafeRoute India — Build & Explore Risk Graph

This notebook walks through building the risk-weighted road graph for a single city  
and visualises crime heatmaps, accident blackspots, and 3-route comparisons.

**Run cells top-to-bottom after completing the data pipeline.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # project root

import pandas as pd
import geopandas as gpd
import numpy as np
import osmnx as ox
import folium
from folium.plugins import HeatMap
from IPython.display import display

from config import TARGET_CITIES, PROCESSED_DIR, GRAPHS_DIR
from src.routing import find_safe_routes, get_route_coordinates

CITY = 'chennai'  # ← change this
print(f'City: {CITY}')

## 1. Explore Crime Data

In [ ]:
crime = pd.read_csv(f'{PROCESSED_DIR}/crime_geocoded.csv')
print(f'Crime records: {len(crime)}')
print(crime.describe())
crime.head()

## 2. Crime Heatmap

In [ ]:
center = TARGET_CITIES[CITY]['center']
m = folium.Map(location=center, zoom_start=11, tiles='CartoDB dark_matter')

score_col = next((c for c in crime.columns if 'NORM' in c.upper()), None)
if score_col:
    heat_data = crime[['LAT', 'LON', score_col]].dropna().values.tolist()
    HeatMap(heat_data, radius=20, blur=15, min_opacity=0.3).add_to(m)

display(m)

## 3. Load Risk Graph and Inspect Edge Scores

In [ ]:
G = ox.load_graphml(f'{GRAPHS_DIR}/{CITY}_final_graph.graphml')
nodes, edges = ox.graph_to_gdfs(G)

score_cols = ['crime_score', 'accident_score', 'flood_score', 'composite_risk', 'risk_prob_high']
for col in score_cols:
    if col in edges.columns:
        vals = edges[col].astype(float)
        print(f'{col:<22} mean={vals.mean():.3f}  max={vals.max():.3f}  '
              f'pct_high={100*(vals>0.55).mean():.1f}%')

## 4. Risk-Coloured Map

In [ ]:
import branca.colormap as cm

colormap = cm.LinearColormap(
    colors=['#22c55e', '#f59e0b', '#ef4444'],
    vmin=0, vmax=1,
    caption='Composite Risk Score'
)

m2 = folium.Map(location=center, zoom_start=13, tiles='CartoDB dark_matter')

for _, edge in edges.iterrows():
    risk = float(edge.get('composite_risk', 0))
    coords = [(c[1], c[0]) for c in edge['geometry'].coords]  # (lat, lon)
    folium.PolyLine(
        coords,
        color=colormap(risk),
        weight=2,
        opacity=0.7,
    ).add_to(m2)

colormap.add_to(m2)
display(m2)

## 5. Route Comparison

In [ ]:
test_pairs = {
    'chennai':   ('Chennai Central', 'T. Nagar'),
    'mumbai':    ('CST Mumbai', 'Dadar'),
    'delhi':     ('Connaught Place', 'India Gate'),
    'bengaluru': ('Majestic', 'Indiranagar'),
    'hyderabad': ('Charminar', 'Banjara Hills'),
}

origin, dest = test_pairs[CITY]
print(f'Finding routes: {origin} → {dest}')

routes, orig, dest_c = find_safe_routes(origin, dest, CITY, travel_hour=20)
print(f'Origin:      {orig}')
print(f'Destination: {dest_c}\n')

for name, r in routes.items():
    if r:
        s = r['summary']
        print(f'{name.upper()}')
        print(f'  Distance:   {s["distance_km"]} km')
        print(f'  Safety:     {s["safety_pct"]}%  ({s["risk_label"]})')
        print(f'  Crime:      {s["avg_crime_score"]}')
        print(f'  Accident:   {s["avg_accident_score"]}')
        print(f'  Flood:      {s["avg_flood_score"]}\n')

In [ ]:
STYLE = {
    'safest':   {'color':'#22c55e','weight':5,'opacity':0.9},
    'balanced': {'color':'#f59e0b','weight':4,'opacity':0.8},
    'fastest':  {'color':'#ef4444','weight':3,'opacity':0.7},
}

m3 = folium.Map(location=orig, zoom_start=13, tiles='CartoDB dark_matter')

for name, r in routes.items():
    if r:
        folium.PolyLine(r['coordinates'], **STYLE[name],
            tooltip=f"{name}: {r['summary']['safety_pct']}% safe"
        ).add_to(m3)

folium.CircleMarker(orig,   radius=8, color='#22c55e', fill=True, tooltip='Origin').add_to(m3)
folium.CircleMarker(dest_c, radius=8, color='#ef4444', fill=True, tooltip='Destination').add_to(m3)

display(m3)